In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [8]:
df = pd.read_csv("covid_toy.csv")

In [9]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [10]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [11]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [13]:
X_train

,age,gender,fever,cough,city
65,69,Female,102.0,Mild,Bangalore
44,20,Male,102.0,Strong,Delhi
96,51,Female,101.0,Strong,Kolkata
10,75,Female,NaN,Mild,Delhi
8,19,Female,100.0,Strong,Bangalore
...,...,...,...,...,...
11,65,Female,98.0,Mild,Mumbai
62,56,Female,104.0,Strong,Bangalore
56,71,Male,NaN,Strong,Kolkata
38,49,Female,101.0,Mild,Delhi


## Aam ziindagi

In [15]:
#assinf simple imputer to fill missing values in the fever column
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

#also the test data
X_test_fever = si.transform(X_test[['fever']])


In [18]:
#OrdinalEncoding -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

#also the test data
X_test_cough = oe.transform(X_test[['cough']])


In [23]:
# OnehotEncoding -> gender
one = OneHotEncoder( drop='first',sparse_output=False)
X_train_gender = one.fit_transform(X_train[['gender', 'city']])

#also the test data
X_test_gender_city = one.fit_transform(X_test[['gender', 'city']])

X_test_gender_city.shape

(20, 4)

In [24]:
#Extracting Age
X_train_age = X_train.drop(columns=['fever', 'cough', 'gender', 'city']).values

#also the test data
X_test_age = X_test.drop(columns=['fever', 'cough', 'gender', 'city']).values

In [26]:
X_train_transformed = np.concatenate([X_train_fever, X_train_cough, X_train_gender, X_train_age], axis=1)

#also the test data
X_test_transformed = np.concatenate([X_test_fever, X_test_cough, X_test_gender_city, X_test_age], axis=1)

X_train_transformed.shape

(80, 7)

it was a heactic job (abhi to sirf 4 the col..agar jyada hue tooo?)

## Mentos Zindagi

In [27]:
from sklearn.compose import ColumnTransformer

In [28]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='passthrough'
) 

In [29]:
transformer.fit_transform(X_train)

array([[102.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  69.        ],
       [102.        ,   1.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  20.        ],
       [101.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  51.        ],
       [100.78082192,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  19.        ],
       [ 99.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  66.        ],
       [101.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  68.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  27.        ],
       [ 98.        ,   0.        ,   0.        ,   1.        ,
          0.    

In [31]:
transformer.fit_transform(X_train).shape

(80, 7)

In [32]:
transformer.transform(X_test).shape

(20, 7)